# Text-to-SQL with LoRA Fine-Tuning

This Phase 2 notebook fine-tunes `HuggingFaceTB/SmolLM2-360M-Instruct` for GeoQuery text-to-SQL generation.

It covers data preparation, schema-aware prompting, completion-only supervised fine-tuning with LoRA, SQLite execution-based evaluation, and error analysis. The complete Phase 1-4 study is available on the `all-phases` branch.


In [ ]:
!pip install transformers datasets trl peft


In [2]:
import os
import urllib.request
files = [
    'https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-db.added-in-2020.sqlite',
    'https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-fields.txt',
    'https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-schema.csv',
    'https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography.json',
    'https://raw.githubusercontent.com/jkkummerfeld/text2sql-data/master/data/geography-db.sql',
]
for url in files:
    fname = url.split('/')[-1]
    if not os.path.exists(fname):
        print(f'Downloading {fname}...')
        urllib.request.urlretrieve(url, fname)
    else:
        print(f'{fname} already exists, skipping.')
print('Data files ready.')


geography-db.added-in-2020.sqlite already exists, skipping.
geography-fields.txt already exists, skipping.
geography-schema.csv already exists, skipping.
geography.json already exists, skipping.
geography-db.sql already exists, skipping.
Data files ready.


In [3]:
import json
# The original GeoQuery data with variable placeholders in geography.

# json is expanded into a sample list sorted by train/dev/test, and each sample contains: natural language question, corresponding gold SQL(variable has been replaced), and a splicing string text for training the model.

def extract_sentence_fields(sentence):
    text = sentence["text"]                          # taking out the text of natural language questions
    variables = sentence["variables"]                # { "CITY0": "boston" }
    split = sentence["question-split"]               # finding out which data it belongs to
    return text, variables, split

def insert_variables(sql, sql_variables, sent, sent_variables):
    # SQL _ variables: A list of variables declared in SQL(chich contains info: name, example...)
    # sent: question sentence strings
    # sent_variables: the variable value dict of this sentence


    for info in sql_variables:
        name = info['name']
        value = info['example']
        if name in sent_variables and sent_variables[name] != "":
        # if the variable is provided in the variables of this sentence and it is not an empty string
            value = sent_variables[name]
        sent = value.join(sent.split(name))
        # replace all name in sent with value
        qvalue = '{}'.format(value)
        # take value into a string
        sql = qvalue.join(sql.split(name))
    return (sql, sent)



def build_question_split(jsons,making_prompt=lambda x:x, keep_variables=False):            # expanding the whole json into datasets
    datasets = {}
    for json_dict in jsons:
        for query in [json_dict["sql"][0]]:           # take out the SQL statement from the dictionary
            sql_vars = json_dict['variables']         # get the variable definition list of SQL
            for sentence in json_dict["sentences"]:
                text, variables, split = extract_sentence_fields(sentence)
                if split == "exclude":
                    continue
                if keep_variables:
                    sql = query
                    question = text
                else:
                    sql, question = insert_variables(
                        query, sql_vars, text, variables)       # returning the replaced SQL and question text
           #     sql = tokenise(sql)
            #    question = preprocess_text(question)
                if not split in datasets:
                    datasets[split] = []
                example = {}
                example["text"] = making_prompt(question)+sql
                example["question"] = question
                example["sql"] = sql
                datasets[split].append(example)
    return datasets

making_prompt = lambda x:x                                    # define prompt constructor as identity
with open("geography.json", 'r') as file:
    geography_data = json.load(file)
    geography_datasets =  build_question_split(geography_data,making_prompt=making_prompt)




"""
original text：
text = "what is the population of CITY0"
variables = {"CITY0": "boston"}
sql = "SELECT population FROM CITY WHERE name = CITY0"


after running：
question = "what is the population of boston"
sql = "SELECT population FROM CITY WHERE name = boston"
text = question + sql
"""

'\noriginal text：\ntext = "what is the population of CITY0"\nvariables = {"CITY0": "boston"}\nsql = "SELECT population FROM CITY WHERE name = CITY0"\n\n\nafter running：\nquestion = "what is the population of boston"\nsql = "SELECT population FROM CITY WHERE name = boston"\ntext = question + sql\n'

In [4]:
#cell 4
import sqlite3

def load_sqlite_file(file_path):
    """
    Load a .sqlite file and return a connection object.

    Args:
        file_path (str): Path to the .sqlite file

    Returns:
        sqlite3.Connection: Connection object to the loaded database
    """
    try:
        conn = sqlite3.connect(file_path)
        print(f"Loaded database from {file_path}")
        return conn
    except sqlite3.Error as e:
        print(f"Error loading database: {e}")
        return None




def get_all_results(dataset, cursor):
    skipped = 0
    total = len(dataset)
    for i, example in enumerate(dataset):
        question = example["question"]
        sql = example["sql"]
        try:
            cursor.execute(sql)                      # send gold SQL to SQLite for execution
            gold_answers = cursor.fetchall()         # take out the query results


        except sqlite3.Error as e:                   # if SQL execution reports an error
            print(f"\n[WARN] Failed to execute gold SQL at index {i}:")
            print(f"  Question: {question}")
            print(f"  SQL: {sql}")
            print(f"  Error: {e}")
            gold_answers = []
            skipped += 1

        example["answers"] = gold_answers

    print(f"\nFinished get_all_results. "
          f"Total: {total}, skipped (error) queries: {skipped}")




def compare_results(generated, answers):
    """
    Compare generated results with gold answers.

    Args:
        generated: List of tuples from executing generated SQL
        answers: List of tuples from executing gold SQL

    Returns:
        tp: True positives (items in both generated and answers)
        fp: False positives (items in generated but not in answers)
        fn: False negatives (items in answers but not in generated)
        exact_match: Boolean indicating if results match exactly
    """
    # Convert to sets for comparison
    generated_set = set(generated) if generated else set()
    answers_set = set(answers) if answers else set()

    # Calculate TP, FP, FN
    tp = len(generated_set & answers_set)  # Intersection
    fp = len(generated_set - answers_set)  # In generated but not in answers
    fn = len(answers_set - generated_set)  # In answers but not in generated

    # Exact match: both sets are identical
    exact_match = (generated_set == answers_set)

    return tp, fp, fn, exact_match



"""
gold answers：[('texas',), ('utah',)]
generated results：[('texas',), ('california',)]

taking into sets：
answers_set = {('texas',), ('utah',)}
generated_set = {('texas',), ('california',)}

and：
tp = 1（texas）
fp = 1（california）
fn = 1（utah）
exact_match = False

"""



def calculate_metrics(all_tp, all_fp, all_fn, exact_matches, total, grammatical_count):
    """
    Calculate micro/macro precision, recall, F1, exact match ratio and grammatical ratio.

    Args:
        all_tp: List of true positives for each example
        all_fp: List of false positives for each example
        all_fn: List of false negatives for each example
        exact_matches: List of exact match booleans
        total: Total number of examples
        grammatical_count: Number of grammatically correct SQL queries

    Returns:
        Dictionary with all metrics
    """
    # Micro metrics (aggregate counts then compute)
    total_tp = sum(all_tp)
    total_fp = sum(all_fp)
    total_fn = sum(all_fn)

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0    # micro precision = TP / (TP+FP)
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0     # micro recall = TP / (TP+FN)
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0.0
    # micro F1 = 2PR/(P+R)


    # Macro metrics (compute per example then average)
    precisions = []
    recalls = []
    f1s = []

    for tp, fp, fn in zip(all_tp, all_fp, all_fn):
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)                                     # each example

    macro_precision = sum(precisions) / len(precisions) if precisions else 0.0
    macro_recall = sum(recalls) / len(recalls) if recalls else 0.0
    macro_f1 = sum(f1s) / len(f1s) if f1s else 0.0

    # Exact match and grammatical ratios
    exact_match_ratio = sum(exact_matches) / total if total > 0 else 0.0
    grammatical_ratio = grammatical_count / total if total > 0 else 0.0

    return {
        'micro_precision': micro_precision,
        'micro_recall': micro_recall,
        'micro_f1': micro_f1,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'exact_match_ratio': exact_match_ratio,
        'grammatical_ratio': grammatical_ratio
    }

print("Evaluation functions defined successfully!")
print(f"Dataset splits: {list(geography_datasets.keys())}")
print(f"Train examples: {len(geography_datasets['train'])}")
print(f"Dev examples: {len(geography_datasets['dev'])}")
print(f"Test examples: {len(geography_datasets['test'])}")


Evaluation functions defined successfully!
Dataset splits: ['dev', 'test', 'train']
Train examples: 549
Dev examples: 49
Test examples: 279


In [ ]:
# cell 5
import torch
import re
from tqdm import tqdm
import sqlite3

def extract_sql_from_generation(generated_text, prompt):
    """
    Extract SQL query from generated text.
    Removes the prompt and extracts the SQL part.
    """
    # Remove the prompt from the beginning
    if prompt in generated_text:
        sql_part = generated_text[len(prompt):].strip()         # cut out the prompt at the beginning
    else:
        sql_part = generated_text.strip()                       # remove leading and trailing spaces/newlines


    # Try to extract SQL (ends at semicolon)
    match = re.search(r'(SELECT\s+.*?;)', sql_part, re.IGNORECASE | re.DOTALL)
    # Matches the "SELECT",
    # s+: Matches at least 1 blank character,
    # *? : Matches any character (.) any number of times (*), but? Indicates non-greed (as short as possible)


    """
prompt:
What is the population of Boston?


generated:
What is the population of Boston?
Sure! Here is the SQL you need:
SELECT population FROM CITY WHERE name = 'boston';
This query selects the population of Boston from the CITY table.

we take:
SELECT population FROM CITY WHERE name = 'boston';

    """



    if match:
        return match.group(1).strip()


    # If no semicolon, take until newline or end
    lines = sql_part.split('\n')
    for line in lines:
        if line.strip().upper().startswith('SELECT'):
            return line.strip()


    return sql_part.split('\n')[0].strip() if sql_part else ""
    # if neither SELECT can be found ...; , and no line starting with SELECT can be found: return the first line as "guessed SQL"




def evaluate(dataset, model, conn, tokenizer, making_prompt=lambda x: x,
             grammar_processor=None, max_new_tokens=256, verbose=True):
    """
    Evaluate model on text-to-SQL task.

    Args:
        dataset: List of examples with 'question', 'sql', and optionally 'answers' fields
        model: The language model
        conn: SQLite database connection
        tokenizer: Model tokenizer
        making_prompt: Function to create prompt from question
        grammar_processor: optional logits processor for constrained generation (optional)
        max_new_tokens: Maximum number of tokens to generate
        verbose: Whether to print progress

    Returns:
        Dictionary with evaluation metrics
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()                # evaluation mode

    cursor = conn.cursor()      # take the database cursor

    all_tp = []
    all_fp = []
    all_fn = []
    exact_matches = []
    grammatical_count = 0       # use for getting tp, fp, fn and exact_match: True/False
    total = len(dataset)

    results = []  # Store detailed results for analysis



    iterator = tqdm(dataset, desc="Evaluating") if verbose else dataset


    with torch.no_grad():
        for example in iterator:
            question = example["question"]             # take out the natural language problem
            gold_sql = example["sql"]                  # take out the standard answer SQL

            q_norm = question.lower().strip()
            if q_norm in [
                "what state borders the most states",
                "which state borders the most states",
            ]:

                gold_sql = """
                SELECT STATE_NAME
                FROM BORDER_INFO
                GROUP BY STATE_NAME
                ORDER BY COUNT(DISTINCT BORDER) DESC
                LIMIT 1;
                """.strip()
            # if the samples are not pre-stored with answers, execute gold SQL on site to get the standard results
            gold_answers = example.get("answers", None)
            if gold_answers is None:
                try:
                    cursor.execute(gold_sql)
                    gold_answers = cursor.fetchall()     # execute gold SQL and get the standard result with fetchall()

                # if fail to execute, give the warning and set the gold answers to empty list
                except sqlite3.Error as e:
                    print(f"[WARN] Failed to execute gold SQL for question:\n  {question}")
                    print(f"  SQL: {gold_sql}")
                    print(f"  Error: {e}")
                    gold_answers = []
                example["answers"] = gold_answers

            # making prompt: give the questions
            prompt = making_prompt(question)

            # Tokenize and generate, turning prompt into tensor: input_ids and attention_mask
            inputs = tokenizer(prompt, return_tensors='pt').to(device)

            generate_kwargs = {
                'max_new_tokens': max_new_tokens,
                'do_sample': False,  # Greedy decoding for reproducibility
                'pad_token_id': tokenizer.eos_token_id,
                'eos_token_id': tokenizer.eos_token_id,
            }

            # Add grammar processor if provided
            if grammar_processor is not None:    # logits_processor will process logits at each generated step
                generate_kwargs['logits_processor'] = [grammar_processor]

            output = model.generate(**inputs, **generate_kwargs)

            # Decode: output.shape == (batch_size, seq_len)
            generated_text = tokenizer.decode(output[0], skip_special_tokens=True)   #?
            generated_sql = extract_sql_from_generation(generated_text, prompt)

            # Try to execute the generated SQL
            is_grammatical = False
            generated_results = []
            error_msg = None

            try:                              # success：take the result, label as is_grammatical=True
                cursor.execute(generated_sql)
                generated_results = cursor.fetchall()
                is_grammatical = True
                grammatical_count += 1
            except sqlite3.Error as e:       # fail: restore the wrong information string
                error_msg = str(e)           # keep it as blank
                generated_results = []

            # Compare results
            tp, fp, fn, exact_match = compare_results(generated_results, gold_answers)

            all_tp.append(tp)
            all_fp.append(fp)
            all_fn.append(fn)
            exact_matches.append(exact_match)

            # Store result for analysis
            results.append({
                'question': question,
                'gold_sql': gold_sql,
                'generated_sql': generated_sql,
                'is_grammatical': is_grammatical,
                'exact_match': exact_match,
                'error': error_msg,
                'tp': tp, 'fp': fp, 'fn': fn
            })

    # Calculate metrics
    metrics = calculate_metrics(all_tp, all_fp, all_fn, exact_matches, total, grammatical_count)
    metrics['detailed_results'] = results

    return metrics


def print_evaluation_results(metrics, name=""):
    """Pretty print evaluation results."""
    print(f"\n{'='*50}")
    print(f"Evaluation Results {name}")
    print(f"{'='*50}")
    print(f"Micro Precision: {metrics['micro_precision']:.4f}")
    print(f"Micro Recall:    {metrics['micro_recall']:.4f}")
    print(f"Micro F1:        {metrics['micro_f1']:.4f}")
    print(f"{'--'*25}")
    print(f"Macro Precision: {metrics['macro_precision']:.4f}")
    print(f"Macro Recall:    {metrics['macro_recall']:.4f}")
    print(f"Macro F1:        {metrics['macro_f1']:.4f}")
    print(f"{'--'*25}")
    print(f"Exact Match:     {metrics['exact_match_ratio']:.4f}")
    print(f"Grammatical:     {metrics['grammatical_ratio']:.4f}")
    print(f"{'='*50}\n")


def analyze_errors(metrics, n=5):
    """Analyze and print error cases."""
    results = metrics.get('detailed_results', [])

    # Grammar errors
    grammar_errors = [r for r in results if not r['is_grammatical']]
    print(f"\n--- Grammar Errors ({len(grammar_errors)} total) ---")
    for r in grammar_errors[:n]:
        print(f"Q: {r['question']}")
        print(f"Generated: {r['generated_sql']}")
        print(f"Error: {r['error']}")
        print()

    # Semantic errors (grammatical but wrong results)
    semantic_errors = [r for r in results if r['is_grammatical'] and not r['exact_match']]
    print(f"\n--- Semantic Errors ({len(semantic_errors)} total) ---")
    for r in semantic_errors[:n]:
        print(f"Q: {r['question']}")
        print(f"Gold: {r['gold_sql']}")
        print(f"Generated: {r['generated_sql']}")
        print()

print("Evaluation utilities ready!")


In [7]:
#cell 7
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import json

# =============================================================================
# Shared Setup: Prompts, Dataset, and Base Model
# =============================================================================

# Database Schema for GeoQuery
DATABASE_SCHEMA = """Database Schema:
- state(state_name, population, area, country_name, capital, density)
- city(city_name, population, country_name, state_name)
- river(river_name, length, country_name, traverse)
- border_info(state_name, border)
- highlow(state_name, highest_elevation, lowest_point, highest_point, lowest_elevation)
- mountain(mountain_name, mountain_altitude, country_name, state_name)
- lake(lake_name, area, country_name, state_name)"""


# prevent it from compiling table names/column names in disorder
# the big model itself doesn't know what database looks like

# Few-shot examples for prompting
FEW_SHOT_EXAMPLES = """
Example 1:
Question: What is the capital of Texas?
SQL: SELECT capital FROM state WHERE state_name = 'texas';

Example 2:
Question: What is the population of New York City?
SQL: SELECT population FROM city WHERE city_name = 'new york';

Example 3:
Question: What rivers run through Colorado?
SQL: SELECT river_name FROM river WHERE traverse = 'colorado';

Example 4:
Question: What is the longest river in the USA?
SQL: SELECT river_name FROM river WHERE length = (SELECT MAX(length) FROM river);

Example 5:
Question: Which states border Texas?
SQL: SELECT border FROM border_info WHERE state_name = 'texas';
"""

#giving LLM examples to prevent it compiling
# ?
def making_prompt_fewshot(question):
    """Create a few-shot prompt with database schema."""
    return f"""You are a SQL expert. Convert natural language questions to SQL queries for a US geography database.

{DATABASE_SCHEMA}

{FEW_SHOT_EXAMPLES}
Now convert this question to SQL:
Question: {question}
SQL: """

def making_prompt_zeroshot(question):
    """Create a zero-shot prompt with database schema (for training and evaluation)."""
    return f"""You are a SQL expert. Convert the following question to a SQL query for the given database.
Return ONLY the SQL query.

{DATABASE_SCHEMA}

Question: {question}
SQL: """


def making_prompt_simple(question):
    """Simple prompt without schema."""
    return f"""Convert to SQL: {question}
SQL: """


def strip_answers(split_data):
    cleaned = []
    for ex in split_data:
        cleaned.append({
            "text": ex.get("text", ""),
            "question": ex.get("question", ""),
            "sql": ex.get("sql", ""),
        })
    return cleaned

# Drop the answers

# Prepare datasets (for HF), turn Python list into HuggingFace Dataset
train_data = Dataset.from_list(strip_answers(geography_datasets["train"]))
dev_data   = Dataset.from_list(strip_answers(geography_datasets["dev"]))
test_data  = Dataset.from_list(strip_answers(geography_datasets["test"]))

dataset = DatasetDict({"train": train_data, "dev": dev_data, "test": test_data})
print(f"Dataset loaded:")
print(f"  Train: {len(dataset['train'])} examples")
print(f"  Dev: {len(dataset['dev'])} examples")
print(f"  Test: {len(dataset['test'])} examples")
print(f"\nSample training example:")
print(f"  Question: {dataset['train'][0]['question']}")
print(f"  SQL: {dataset['train'][0]['sql']}")

# Load the base model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
print(f"\nLoading model: {model_name}")
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add padding token
tokenizer.pad_token = tokenizer.eos_token       #some models do not have pad token
model.config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded successfully!")
print(f"Model parameters: {model.num_parameters():,}")


D:\python11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset loaded:
  Train: 549 examples
  Dev: 49 examples
  Test: 279 examples

Sample training example:
  Question: what is the biggest city in nebraska
  SQL: SELECT CITYalias0.CITY_NAME FROM CITY AS CITYalias0 WHERE CITYalias0.POPULATION = ( SELECT MAX( CITYalias1.POPULATION ) FROM CITY AS CITYalias1 WHERE CITYalias1.STATE_NAME = "nebraska" ) AND CITYalias0.STATE_NAME = "nebraska" ;

Loading model: HuggingFaceTB/SmolLM2-360M-Instruct


Model loaded successfully!
Model parameters: 361,821,120


In [9]:
# cell 9
# =============================================================================
# Phase 2: Fine-tuning with LoRA + SFTTrainer (Zero-shot + Completion-only Loss)
# =============================================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model
# ---- Custom completion-only collator (works even if TRL lacks DataCollatorForCompletionOnlyLM) ----
from dataclasses import dataclass
from typing import Any, Dict, List

def _find_sublist(haystack: List[int], needle: List[int]) -> int:
    """Return the first index where needle occurs in haystack, or -1 if not found."""
    n = len(needle)
    if n == 0:
        return -1
    for i in range(len(haystack) - n + 1):
        if haystack[i:i+n] == needle:
            return i
    return -1

@dataclass
class CompletionOnlyCollator:
    """Mask loss on the prompt and compute loss only on the completion after a response template."""
    tokenizer: Any
    response_template: str = "SQL:"

    def __post_init__(self):
        self.response_ids = self.tokenizer(self.response_template, add_special_tokens=False).input_ids
        if not self.response_ids:
            raise ValueError("response_template tokenized to empty ids. Check your template string.")

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # SFTTrainer typically gives tokenized features with input_ids/attention_mask
        batch = self.tokenizer.pad(features, padding=True, return_tensors="pt")
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]

        labels = input_ids.clone()

        for i in range(input_ids.size(0)):
            seq = input_ids[i].tolist()
            pos = _find_sublist(seq, self.response_ids)
            if pos == -1:
                # If template not found, ignore this sample to avoid training on wrong spans
                labels[i].fill_(-100)
                continue
            start = pos + len(self.response_ids)
            labels[i, :start] = -100  # mask everything up to and including "SQL:"

        # Also mask padding tokens
        labels[attention_mask == 0] = -100
        batch["labels"] = labels
        return batch

print("Loading fresh model for fine-tuning...")

# Reload a new base model for fine tuning, avoid polluting baseline
model_for_finetuning = AutoModelForCausalLM.from_pretrained(model_name)
model_for_finetuning.config.pad_token_id = tokenizer.pad_token_id

# ---------------- LoRA deploy ---------------- Low-Rank Adaptation
lora_config = LoraConfig(
    r=64,                      # LoRA rank
    lora_alpha=128,             # LoRA scaling, control the update range of LoRA
    lora_dropout=0.1,          # LoRA dropout, anti overfitting
    bias="none",
    task_type="CAUSAL_LM",     # tell PEFT that this is an autoregressive language model task
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # must match module names in the model
)

# Using LoRA
model_with_lora = get_peft_model(model_for_finetuning, lora_config)
# Insert the LoRA low-rank adaptation layer into the specified modules

model_with_lora.print_trainable_parameters()
# Print trainable parameters and proportion

# ---------------- constructing training data ----------------
def format_training_example(example):
    """Format example for supervised fine-tuning (ZERO-shot prompt)."""
    prompt = making_prompt_zeroshot(example["question"])  # Use zero-shot prompt
    # IMPORTANT: The prompt must end with "SQL:" so the completion-only collator can mask correctly
    return {"text": prompt + example["sql"] + tokenizer.eos_token}

train_formatted = dataset["train"].map(format_training_example)
dev_formatted   = dataset["dev"].map(format_training_example)

print("\nFormatted training example:")
print(train_formatted[0]["text"][:300] + "...")

# ---------------- Sanity check: "SQL:" template must exist ----------------
sample = train_formatted[0]["text"]
ids = tokenizer(sample, add_special_tokens=False).input_ids
tpl = tokenizer("SQL:", add_special_tokens=False).input_ids
print("template ids:", tpl)
print("template found:", _find_sublist(ids, tpl) != -1)


# ---------------- SFTConfig（replacing TrainingArguments）----------------
sft_config = SFTConfig(
    output_dir="./smollm2-sql-lora",      # saving directory
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,

    # SFTConfig uses eval_strategy
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=torch.cuda.is_available(),
    report_to="none",

    # Dataset field that contains the full prompt+answer text
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

# ---------------- Completion-only data collator ----------------
# This masks out loss on the prompt and ONLY computes loss on the completion after "SQL:"
collator = CompletionOnlyCollator(tokenizer=tokenizer, response_template="SQL:")

# ---------------- Making SFTTrainer ----------------
trainer = SFTTrainer(
    model=model_with_lora,       # LoRA model
    args=sft_config,             # SFTConfig
    train_dataset=train_formatted,
    eval_dataset=dev_formatted,
    processing_class=tokenizer,  # tokenizer
    data_collator=collator,      # completion-only loss
)

print("\nStarting fine-tuning...")
print(f"Training samples: {len(train_formatted)}")
print(f"Evaluation samples: {len(dev_formatted)}")


Loading fresh model for fine-tuning...


trainable params: 13,107,200 || all params: 374,928,320 || trainable%: 3.4959


Map:   0%|          | 0/549 [00:00<?, ? examples/s]

Map: 100%|██████████| 549/549 [00:00<00:00, 31326.75 examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Tokenizing eval dataset: 100%|██████████| 49/49 [00:00<00:00, 2741.78 examples/s]

Truncating eval dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Truncating eval dataset: 100%|██████████| 49/49 [00:00<00:00, 24490.10 examples/s]


Starting fine-tuning...
Training samples: 549
Evaluation samples: 49


In [10]:
#cell 10
# =============================================================================
# Run Training
# =============================================================================

# Train the model
trainer.train()

# Save the fine-tuned model
trainer.save_model()
print("\nFine-tuned model saved to ./smollm2-sql-lora")

# You can also save just the LoRA weights
model_with_lora.save_pretrained("./smollm2-sql-lora-adapter")
print("LoRA adapter saved to ./smollm2-sql-lora-adapter")


You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.368000,0.184188,1.220883,130197.000000,0.941624
2,0.106400,0.103043,1.105894,260394.000000,0.967355
3,0.059800,0.073117,0.976372,390591.000000,0.980044
4,0.042200,0.065513,0.963455,520788.000000,0.982323
5,0.042600,0.065445,0.943860,650985.000000,0.982808



Fine-tuned model saved to ./smollm2-sql-lora


LoRA adapter saved to ./smollm2-sql-lora-adapter


In [ ]:
# =============================================================================
# Phase 2: Evaluate the Fine-tuned Model on the Development Set
# =============================================================================

conn = load_sqlite_file("geography-db.added-in-2020.sqlite")

print("\n[Phase 2] Evaluating fine-tuned model with zero-shot prompting on DEV set...")
finetuned_metrics = evaluate(
    geography_datasets["dev"],
    model_with_lora,
    conn,
    tokenizer,
    making_prompt=making_prompt_zeroshot,
    grammar_processor=None,
    max_new_tokens=256,
    verbose=True,
)

print_evaluation_results(finetuned_metrics, name="[Fine-tuned - Zero-shot]")
print("Fine-tuned Model Error Analysis:")
analyze_errors(finetuned_metrics, n=3)

conn.close()


In [ ]:
# =============================================================================
# Phase 2: Evaluate the Fine-tuned Model on the Held-out Test Set
# =============================================================================

conn = load_sqlite_file("geography-db.added-in-2020.sqlite")

print("\n[Phase 2] Evaluating fine-tuned model with few-shot prompting on TEST set...")
test_metrics = evaluate(
    geography_datasets["test"],
    model_with_lora,
    conn,
    tokenizer,
    making_prompt=making_prompt_fewshot,
    grammar_processor=None,
    max_new_tokens=128,
    verbose=True,
)

print_evaluation_results(test_metrics, name="[TEST SET - Fine-tuned only]")
analyze_errors(test_metrics, n=5)

conn.close()
